<a href="https://colab.research.google.com/github/petersonrs/petersonrs/blob/main/NIVEL_RIOS_V2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

"""
Monitor de nível de rio - Defesa Civil de Itajaí (e outros municípios via
o parâmetro municipio_id: 1=Itajaí, 2=Brusque, 3=Blumenau, 4=Rio do Sul).

ESTRUTURA DA PÁGINA
--------------------
Cada estação aparece dentro de um <section class="cartao">, com o nome/
código da estação no <header><h2>...</h2></header>. Dentro da mesma seção
podem existir até duas tabelas:

  1) "Consultar dados do gráfico" - tabela simples (Estação, Data e hora,
     Nível do Rio). Aparece em todos os municípios. Cada linha já traz o
     nome da estação, então essa tabela é auto-suficiente.

  2) "Histórico de dados da estação" - tabela mais completa, só em alguns
     municípios (ex: Itajaí), com nível, variação (subida/descida) e chuva
     em várias janelas (10min, 1h, 6h, 12h, 24h, 48h). Essa tabela NÃO tem
     coluna de estação - por isso usamos o <h2> da seção-pai para saber de
     qual estação são os dados.

Como é tudo SPA (JS), o Selenium só renderiza a página; o BeautifulSoup
faz toda a extração sobre o HTML já pronto.

INSTALAÇÃO
----------
pip install -r requirements.txt
(é necessário ter o Google Chrome instalado na máquina)

USO
---
python monitor_rios_itajai_v4.py

Gera, dentro de ./historico_rios/:
  - nivel_rio_<codigo>.parquet         (da tabela simples, todas as estações)
  - dados_completos_<codigo>.parquet   (da tabela detalhada, quando existir)
"""

In [50]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [51]:
!pip install selenium>=4.20.0

# Instalação do webdriver_manager para gerenciar o chromedriver (se já não estiver instalado)
!pip install webdriver_manager

# Instala o Chrome no ambiente Colab, essencial para o Selenium com ChromeDriver
!apt-get update
!wget -q -O - https://dl-ssl.google.com/linux/linux_signing_key.pub | apt-key add -
!echo "deb [arch=amd64] http://dl.google.com/linux/chrome/deb/ stable main" >> /etc/apt/sources.list.d/google-chrome.list
!apt-get update
!apt-get install -y google-chrome-stable

Hit:1 http://dl.google.com/linux/chrome/deb stable InRelease
Hit:2 https://dl.google.com/linux/chrome-stable/deb stable InRelease
Hit:3 https://cli.github.com/packages stable InRelease
Hit:4 https://cloud.r-project.org/bin/linux/ubuntu noble-cran40/ InRelease
Hit:5 http://archive.ubuntu.com/ubuntu noble InRelease
Hit:6 http://archive.ubuntu.com/ubuntu noble-updates InRelease
Hit:7 http://security.ubuntu.com/ubuntu noble-security InRelease
Hit:8 http://archive.ubuntu.com/ubuntu noble-backports InRelease
Hit:9 https://r2u.stat.illinois.edu/ubuntu noble InRelease
Hit:10 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu noble InRelease
Reading package lists... Done
W: http://dl.google.com/linux/chrome/deb/dists/stable/InRelease: Key is stored in legacy trusted.gpg keyring (/etc/apt/trusted.gpg), see the DEPRECATION section in apt-key(8) for details.
W: Target Packages (main/binary-amd64/Packages) is configured multiple times in /etc/apt/sources.list.d/google-chrome.list:1 and /etc/apt

In [52]:
import re
import sys
import logging
from datetime import datetime
from pathlib import Path

import pandas as pd
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.common.exceptions import TimeoutException

In [62]:
# ------------------------- CONFIGURAÇÃO -------------------------

URLS = {
    "itajai": "https://monitoramento.defesacivil.itajai.sc.gov.br/monitoramento/rios",
    "brusque": "https://monitoramento.defesacivil.itajai.sc.gov.br/monitoramento/rios?municipio_id=2",
}

# Trechos que identificam as estações que você quer salvar.
ESTACOES_ALVO = ["DC01","DC02","DC03","DC04","DC05","DC06","DC07","DC08","DC09","DC10","DC11","DCSC-00019"]

# Onde o histórico será salvo (um arquivo parquet por estação)
OUTPUT_DIR = Path("/content/drive/MyDrive/Python/Analise Volume Rio")

TIMEOUT_CARREGAMENTO = 25

REGEX_CODIGO = re.compile(r"(DC\d{2}|DCSC-\d+)")
REGEX_NUM = re.compile(r"-?\d+(?:,\d+)?")

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
)
log = logging.getLogger(__name__)


def main():
    driver = criar_driver()
    registros_simples, registros_detalhados = [], []

    try:
        for fonte, url in URLS.items():
            log.info("Abrindo [%s] %s ...", fonte, url)
            driver.get(url)
            try:
                esperar_conteudo_carregar(driver)
            except RuntimeError as e:
                log.warning("Pulando '%s': %s", fonte, e)
                continue

            html = driver.page_source
            soup = BeautifulSoup(html, "html.parser")

            simples = extrair_tabela_simples(soup, fonte)
            detalhados = extrair_tabela_detalhada(soup, fonte)
            log.info("[%s] %d leituras simples, %d leituras detalhadas",
                      fonte, len(simples), len(detalhados))

            registros_simples.extend(simples)
            registros_detalhados.extend(detalhados)
    finally:
        driver.quit()

    if not registros_simples and not registros_detalhados:
        raise RuntimeError(
            f"Nenhuma leitura encontrada para as estações {ESTACOES_ALVO}."
        )

    salvar_parquet(registros_simples, "nivel_rio", ["estacao_codigo", "data_hora_medicao"])
    salvar_parquet(registros_detalhados, "dados_completos", ["estacao_codigo", "data_hora_medicao"])


In [63]:
# ------------------------- FUNÇÕES -------------------------
def criar_driver() -> webdriver.Chrome:
    options = Options()
    options.add_argument("--headless=new")
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    options.add_argument("--disable-gpu")
    options.add_argument("--window-size=1920,1080")
    options.add_argument(
        "user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0 Safari/537.36"
    )
    return webdriver.Chrome(options=options)


In [64]:
def esperar_conteudo_carregar(driver):
    wait = WebDriverWait(driver, TIMEOUT_CARREGAMENTO)

    def conteudo_carregado(d):
        body_text = d.find_element(By.TAG_NAME, "body").text
        return any(codigo in body_text for codigo in ESTACOES_ALVO)

    try:
        wait.until(conteudo_carregado)
    except TimeoutException:
        raise RuntimeError(
            "Conteúdo da página não carregou a tempo, ou nenhuma das "
            f"estações {ESTACOES_ALVO} apareceu no texto da página."
        )

In [65]:
def extrair_numero(texto: str):
    """Extrai o primeiro número (com vírgula decimal) de um texto, ou None."""
    m = REGEX_NUM.search(texto)
    return float(m.group().replace(",", ".")) if m else None

In [66]:
def extrair_tabela_simples(soup: BeautifulSoup, fonte: str) -> list[dict]:
    """Tabela 'Consultar dados do gráfico' (Estação, Data e hora, Nível do
    Rio). Auto-suficiente: cada linha já traz o nome da estação."""
    agora = datetime.now()
    registros = []

    for tabela in soup.find_all("table"):
        cabecalho = [th.get_text(strip=True) for th in tabela.select("thead th")]
        if not any("Nível do Rio" in c for c in cabecalho):
            continue

        for linha in tabela.select("tbody tr"):
            celulas = [td.get_text(strip=True) for td in linha.find_all("td")]
            if len(celulas) < 3:
                continue
            estacao_texto, data_hora_texto, nivel_texto = celulas[0], celulas[1], celulas[2]

            if not any(codigo in estacao_texto for codigo in ESTACOES_ALVO):
                continue

            match_codigo = REGEX_CODIGO.search(estacao_texto)
            codigo = match_codigo.group(1) if match_codigo else estacao_texto
            nivel_m = extrair_numero(nivel_texto)
            data_hora_medicao = datetime.strptime(data_hora_texto, "%d/%m/%Y, %H:%M:%S")

            registros.append({
                "estacao_codigo": codigo,
                "estacao_descricao": estacao_texto.strip(),
                "data_hora_medicao": data_hora_medicao,
                "nivel_m": nivel_m,
                "fonte": fonte,
                "capturado_em": agora,
            })

    return registros

In [67]:
def extrair_tabela_detalhada(soup: BeautifulSoup, fonte: str) -> list[dict]:
    """Tabela 'Histórico de dados da estação' (nível + variação + chuva).
    Não tem coluna de estação - usamos o <h2> da seção <section class="cartao">
    que a envolve para identificar de qual estação são os dados."""
    agora = datetime.now()
    registros = []

    for section in soup.select("section.cartao"):
        h2 = section.select_one("header h2")
        nome_estacao = h2.get_text(strip=True) if h2 else None
        if not nome_estacao:
            continue

        match_codigo = REGEX_CODIGO.search(nome_estacao)
        codigo = match_codigo.group(1) if match_codigo else None
        if not codigo or codigo not in ESTACOES_ALVO:
            continue

        for det in section.select("details"):
            summary = det.select_one("summary")
            if not summary or "Histórico de dados da estação" not in summary.get_text():
                continue

            tabela = det.select_one("table")
            if not tabela:
                continue

            for linha in tabela.select("tbody tr"):
                celulas = linha.find_all("td")
                if len(celulas) < 10:
                    continue

                data_hora_texto = celulas[0].get_text(strip=True)
                data_hora_medicao = datetime.strptime(data_hora_texto, "%d/%m/%Y, %H:%M")

                nivel_m = extrair_numero(celulas[1].get_text(" ", strip=True))
                variacao_m = extrair_numero(celulas[2].get_text(strip=True))
                chuva_10min_mm = extrair_numero(celulas[3].get_text(strip=True))
                chuva_mmh = extrair_numero(celulas[4].get_text(strip=True))
                chuva_1h_mm = extrair_numero(celulas[5].get_text(strip=True))
                chuva_6h_mm = extrair_numero(celulas[6].get_text(strip=True))
                chuva_12h_mm = extrair_numero(celulas[7].get_text(strip=True))
                chuva_24h_mm = extrair_numero(celulas[8].get_text(strip=True))
                chuva_48h_mm = extrair_numero(celulas[9].get_text(strip=True))

                registros.append({
                    "estacao_codigo": codigo,
                    "estacao_descricao": nome_estacao,
                    "data_hora_medicao": data_hora_medicao,
                    "nivel_m": nivel_m,
                    "variacao_m": variacao_m,
                    "chuva_10min_mm": chuva_10min_mm,
                    "chuva_mmh": chuva_mmh,
                    "chuva_1h_mm": chuva_1h_mm,
                    "chuva_6h_mm": chuva_6h_mm,
                    "chuva_12h_mm": chuva_12h_mm,
                    "chuva_24h_mm": chuva_24h_mm,
                    "chuva_48h_mm": chuva_48h_mm,
                    "fonte": fonte,
                    "capturado_em": agora,
                })

    return registros

In [68]:
def salvar_parquet(registros: list[dict], prefixo_arquivo: str, chave_dedup: list):
    """Agrupa por estacao_codigo e acrescenta ao Parquet histórico de cada
    uma, evitando duplicar a mesma medição."""
    if not registros:
        return

    df_novos = pd.DataFrame(registros)

    # Obter a data da captura para criar a estrutura de diretórios
    # Assumimos que todos os registros no 'registros' têm o mesmo 'capturado_em'
    data_captura = df_novos['capturado_em'].iloc[0]

    # Criar o caminho dinâmico: OUTPUT_DIR / "parquet" / ano / mes / dia
    dynamic_output_dir = OUTPUT_DIR / "parquet" / str(data_captura.year) / f"{data_captura.month:02d}" / f"{data_captura.day:02d}"
    dynamic_output_dir.mkdir(parents=True, exist_ok=True)

    #OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    for codigo, grupo in df_novos.groupby("estacao_codigo"):
        nome_arquivo = re.sub(r"[^A-Za-z0-9_-]", "_", codigo.lower())
        arquivo = dynamic_output_dir / f"{prefixo_arquivo}_{nome_arquivo}.parquet"

        if arquivo.exists():
            df_existente = pd.read_parquet(arquivo)
            combinado = pd.concat([df_existente, grupo], ignore_index=True)
            combinado = combinado.drop_duplicates(subset=chave_dedup, keep="last")
        else:
            combinado = grupo

        combinado = combinado.sort_values("data_hora_medicao").reset_index(drop=True)
        combinado.to_parquet(arquivo, index=False)

        ultima = combinado.iloc[-1]
        log.info(
            "[%s] %s: %d linhas em %s | última leitura: %s m às %s",
            codigo, prefixo_arquivo, len(combinado), arquivo,
            ultima["nivel_m"], ultima["data_hora_medicao"],
        )

In [69]:
# Criar o diretório de logs se não existir
log_dir = OUTPUT_DIR / "log"
log_dir.mkdir(parents=True, exist_ok=True)

# Caminho do arquivo de log
log_file_path = log_dir / "execucao.log"

# Configurar novamente o logging com múltiplos Handlers (Console + Arquivo)
for handler in logging.root.handlers[:]:
    logging.root.removeHandler(handler)

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    handlers=[
        logging.StreamHandler(sys.stdout),
        logging.FileHandler(log_file_path, encoding="utf-8")
    ]
)

# Redefine o logger principal do notebook para utilizar a nova configuração
log = logging.getLogger(__name__)
log.info("Configuração de log atualizada. Logs serão salvos em: %s", log_file_path)

Exception ignored in: <_io.TextIOWrapper name='/content/drive/MyDrive/Python/Analise Volume Rio/log/execucao.log' mode='a' encoding='utf-8'>
Traceback (most recent call last):
  File "/tmp/ipykernel_464/4186598057.py", line 9, in <cell line: 0>
OSError: [Errno 107] Transport endpoint is not connected


2026-11-09 11:54:21,883 [INFO] Configuração de log atualizada. Logs serão salvos em: /content/drive/MyDrive/Python/Analise Volume Rio/log/execucao.log


In [70]:
if __name__ == "__main__":
    try:
        main()
    except Exception as e:
        log.error("Falha na captura: %s", e)
        sys.exit(1)

2026-11-09 11:54:25,648 [INFO] Abrindo [itajai] https://monitoramento.defesacivil.itajai.sc.gov.br/monitoramento/rios ...
2026-11-09 11:54:29,931 [INFO] [itajai] 0 leituras simples, 220 leituras detalhadas
2026-11-09 11:54:29,935 [INFO] Abrindo [brusque] https://monitoramento.defesacivil.itajai.sc.gov.br/monitoramento/rios?municipio_id=2 ...
2026-11-09 11:54:30,659 [INFO] [brusque] 145 leituras simples, 0 leituras detalhadas
2026-11-09 11:54:30,791 [INFO] [DCSC-00019] nivel_rio: 145 linhas em /content/drive/MyDrive/Python/Analise Volume Rio/parquet/2026/11/09/nivel_rio_dcsc-00019.parquet | última leitura: 2.23 m às 2026-11-09 08:50:00
2026-11-09 11:54:30,835 [INFO] [DC01] dados_completos: 33 linhas em /content/drive/MyDrive/Python/Analise Volume Rio/parquet/2026/11/09/dados_completos_dc01.parquet | última leitura: 0.95 m às 2026-11-09 08:41:00
2026-11-09 11:54:30,891 [INFO] [DC02] dados_completos: 33 linhas em /content/drive/MyDrive/Python/Analise Volume Rio/parquet/2026/11/09/dados_co